# Entropy & KL Divergence

Companion notebook for the [Entropy & KL Divergence](https://ml-viz.vercel.app/courses/probability-statistics/05-entropy-and-kl-divergence) lesson on ML Viz.

We'll compute surprise, entropy, cross-entropy, and KL divergence numerically, verify the identity H(p,q) = H(p) + KL(p‖q), and watch the two KL directions behave differently on a bimodal target.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams["figure.facecolor"] = "#0f1117"
plt.rcParams["axes.facecolor"] = "#1a1d27"
plt.rcParams["axes.edgecolor"] = "#2e3347"
plt.rcParams["grid.color"] = "#2e3347"

def entropy(p):
    p = p[p > 0]
    return -(p * np.log(p)).sum()

def cross_entropy(p, q):
    return -(p[p > 0] * np.log(q[p > 0])).sum()

def kl(p, q):
    mask = p > 0
    return (p[mask] * np.log(p[mask] / q[mask])).sum()

## 1. Entropy across coin biases

Entropy peaks at the fair coin and collapses toward certainty at the edges.

In [ ]:
thetas = np.linspace(0.001, 0.999, 200)
H = [entropy(np.array([t, 1 - t])) for t in thetas]

plt.figure(figsize=(8, 3.5))
plt.plot(thetas, H, color="#6366f1", lw=2)
plt.axvline(0.5, color="#eab308", ls="--", label="fair coin: max entropy")
plt.xlabel("P(heads)")
plt.ylabel("entropy (nats)")
plt.title("Uncertainty is maximal when you know least")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

## 2. The identity H(p, q) = H(p) + KL(p‖q)

Cross-entropy = irreducible uncertainty of the data + your modeling penalty.

In [ ]:
p = np.array([0.7, 0.2, 0.1])
q = np.array([0.5, 0.3, 0.2])

print(f"H(p)      = {entropy(p):.4f}")
print(f"H(p, q)   = {cross_entropy(p, q):.4f}")
print(f"KL(p‖q)   = {kl(p, q):.4f}")
print(f"H + KL    = {entropy(p) + kl(p, q):.4f}   (matches cross-entropy)")
print()
print(f"KL(p‖q) = {kl(p, q):.4f}  vs  KL(q‖p) = {kl(q, p):.4f}   (asymmetric!)")

## 3. Cross-entropy IS the classification loss

A one-hot data distribution collapses cross-entropy to −log q(true class) — the familiar log-loss.

In [ ]:
q_pred = np.array([0.05, 0.85, 0.10])      # model's softmax output
p_true = np.array([0.0, 1.0, 0.0])           # one-hot truth (class 1)

print(f"cross-entropy      : {cross_entropy(p_true, q_pred):.4f}")
print(f"-log q(true class) : {-np.log(q_pred[1]):.4f}")

## 4. Forward vs reverse KL on a bimodal target

Fit a single Gaussian q to a two-bump p by minimizing each direction (crude grid search). Forward KL must cover both bumps; reverse KL happily picks one.

In [ ]:
x = np.linspace(-6, 6, 601)
dx = x[1] - x[0]

def gauss(x, mu, sd):
    g = np.exp(-((x - mu) ** 2) / (2 * sd**2))
    return g / (g.sum() * dx)

p_bimodal = 0.5 * gauss(x, -2, 0.6) + 0.5 * gauss(x, 2, 0.6)
p_disc = p_bimodal * dx                       # discretize to probabilities

best = {}
for direction in ["forward", "reverse"]:
    best_kl, best_q = np.inf, None
    for mu in np.linspace(-3, 3, 61):
        for sd in np.linspace(0.4, 3.5, 32):
            q_disc = gauss(x, mu, sd) * dx
            d = kl(p_disc, q_disc) if direction == "forward" else kl(q_disc, p_disc)
            if d < best_kl:
                best_kl, best_q = d, q_disc
    best[direction] = best_q

plt.figure(figsize=(9, 4))
plt.plot(x, p_disc / dx, color="#94a3b8", lw=2, label="target p (bimodal)")
plt.plot(x, best["forward"] / dx, color="#14b8a6", lw=2, label="argmin KL(p‖q): covers both modes")
plt.plot(x, best["reverse"] / dx, color="#f43f5e", lw=2, label="argmin KL(q‖p): picks one mode")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Same formula, opposite personalities")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Entropy

Entropy is average surprise, in bits:

$$H(p) = -\sum_i p_i \log_2 p_i \qquad (0 \log 0 = 0)$$

Implement it, handling zero probabilities (drop them — their limit contributes nothing). The checks pin down the landmarks: a fair coin is exactly **1 bit**, certainty is **0**, and a uniform choice among 8 options is **3 bits**.

In [ ]:
def entropy(p):
    """Entropy of a discrete distribution, in bits."""
    p = np.asarray(p, dtype=float)

    # TODO(you): keep only the strictly positive entries (0 log 0 = 0)
    p = ...

    # TODO(you): -sum(p * log2(p))
    return ...

In [ ]:
# Checks — run me
assert abs(entropy([0.5, 0.5]) - 1) < 1e-12, "fair coin: exactly 1 bit"
assert abs(entropy([1.0, 0.0])) < 1e-12, "certainty: 0 bits"
assert abs(entropy([1 / 8] * 8) - 3) < 1e-12, "uniform over 8 outcomes: 3 bits"
assert entropy([0.9, 0.1]) < entropy([0.6, 0.4]), "more lopsided -> less entropy"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))
```

</details>

### Exercise 2 — KL divergence and the cross-entropy identity

KL divergence is the *extra* bits paid for coding $p$ with a code built for $q$:

$$\text{KL}(p \,\|\, q) = \sum_i p_i \log_2 \frac{p_i}{q_i}$$

Implement it (sum only where $p_i > 0$). The checks verify the three properties this lesson leans on: $\text{KL}(p\|p) = 0$, KL is **not symmetric**, and the exact identity $H(p, q) = H(p) + \text{KL}(p \,\|\, q)$ from section 2.

In [ ]:
def kl_divergence(p, q):
    """KL(p || q) in bits. Sums only over entries where p > 0."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    # TODO(you): boolean mask of entries where p > 0
    mask = ...

    # TODO(you): sum p * log2(p / q) over the masked entries
    return ...

In [ ]:
# Checks — run me
p = np.array([0.7, 0.2, 0.1])
q = np.array([0.4, 0.4, 0.2])

assert abs(kl_divergence(p, p)) < 1e-12, "KL(p, p) = 0"
assert kl_divergence(p, q) > 0, "KL is nonnegative (Gibbs' inequality)"
assert abs(kl_divergence(p, q) - kl_divergence(q, p)) > 1e-3, "KL is NOT symmetric"

cross_entropy = -np.sum(p * np.log2(q))
assert abs(cross_entropy - (entropy(p) + kl_divergence(p, q))) < 1e-12, \
    "H(p, q) = H(p) + KL(p || q)"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def kl_divergence(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    mask = p > 0
    return np.sum(p[mask] * np.log2(p[mask] / q[mask]))
```

</details>

**Next:** see these losses in the wild — the [course quiz](https://ml-viz.vercel.app/courses/probability-statistics/06-quiz), or jump to [VAEs](https://ml-viz.vercel.app/courses/generative-models/03-variational-autoencoders) where the reverse-KL story plays out in the ELBO.